In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/application_train.csv")

print("Shape:", df.shape)
print("\nFirst 3 rows:")
df.head(3)

Shape: (307511, 122)

First 3 rows:


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# TARGET = 1 means defaulted, 0 means repaid
target_counts = df["TARGET"].value_counts()
target_pct = df["TARGET"].value_counts(normalize=True) * 100

print("Raw counts:\n", target_counts)
print("\nPercentage:\n", target_pct.round(2))

Raw counts:
 TARGET
0    282686
1     24825
Name: count, dtype: int64

Percentage:
 TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64


In [3]:
# These columns map directly to your 3 data input boxes
key_columns = [
    "TARGET",
    
    # Bank cash flow box
    "AMT_INCOME_TOTAL",     # monthly income
    "AMT_CREDIT",           # loan amount requested
    "AMT_ANNUITY",          # monthly repayment amount
    "AMT_GOODS_PRICE",      # price of goods loan is for
    
    # UPI patterns / transaction behaviour box  
    "DAYS_EMPLOYED",        # how long at current job (stability)
    "DAYS_BIRTH",           # age in days (negative number)
    
    # Bill regularity box
    "CNT_FAM_MEMBERS",      # family size (affects expenses)
    "OBS_30_CNT_SOCIAL_CIRCLE",   # people in network who defaulted
]

df[key_columns].describe().round(2)

,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_EMPLOYED,DAYS_BIRTH,CNT_FAM_MEMBERS,OBS_30_CNT_SOCIAL_CIRCLE
count,307511.00,3.075110e+05,307511.00,307499.00,307233.00,307511.00,307511.00,307509.00,306490.00
mean,0.08,1.687979e+05,599026.00,27108.57,538396.21,63815.05,-16037.00,2.15,1.42
std,0.27,2.371231e+05,402490.78,14493.74,369446.46,141275.77,4363.99,0.91,2.40
min,0.00,2.565000e+04,45000.00,1615.50,40500.00,-17912.00,-25229.00,1.00,0.00
25%,0.00,1.125000e+05,270000.00,16524.00,238500.00,-2760.00,-19682.00,2.00,0.00
50%,0.00,1.471500e+05,513531.00,24903.00,450000.00,-1213.00,-15750.00,2.00,0.00
75%,0.00,2.025000e+05,808650.00,34596.00,679500.00,-289.00,-12413.00,3.00,2.00
max,1.00,1.170000e+08,4050000.00,258025.50,4050000.00,365243.00,-7489.00,20.00,348.00


In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

# Show only columns that actually have missing values
print(missing_report[missing_report["missing_count"] > 0].head(20))
print(f"\nTotal columns with missing values: {(missing_report['missing_count'] > 0).sum()}")
print(f"Total columns: {df.shape[1]}")

                          missing_count  missing_pct
COMMONAREA_MEDI                  214865        69.87
COMMONAREA_AVG                   214865        69.87
COMMONAREA_MODE                  214865        69.87
NONLIVINGAPARTMENTS_MODE         213514        69.43
NONLIVINGAPARTMENTS_AVG          213514        69.43
NONLIVINGAPARTMENTS_MEDI         213514        69.43
FONDKAPREMONT_MODE               210295        68.39
LIVINGAPARTMENTS_MODE            210199        68.35
LIVINGAPARTMENTS_AVG             210199        68.35
LIVINGAPARTMENTS_MEDI            210199        68.35
FLOORSMIN_AVG                    208642        67.85
FLOORSMIN_MODE                   208642        67.85
FLOORSMIN_MEDI                   208642        67.85
YEARS_BUILD_MEDI                 204488        66.50
YEARS_BUILD_MODE                 204488        66.50
YEARS_BUILD_AVG                  204488        66.50
OWN_CAR_AGE                      202929        65.99
LANDAREA_MEDI                    182590       

In [5]:
# Same groupby you did in Step 2 — but now on 300k real applicants
summary = df.groupby("TARGET").agg(
    avg_income        = ("AMT_INCOME_TOTAL", "mean"),
    avg_credit        = ("AMT_CREDIT", "mean"),
    avg_annuity       = ("AMT_ANNUITY", "mean"),
    count             = ("TARGET", "count")
).round(2)

print(summary)

        avg_income  avg_credit  avg_annuity   count
TARGET                                             
0        169077.72   602648.28     27163.62  282686
1        165611.76   557778.53     26481.74   24825
